In [75]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

# load .env
load_dotenv()

# load the config
with open("config/config.json", "r") as f:
    config = json.load(f)

print(config["llm"])

{'openai_model': 'gpt-4o-mini', 'gemini_model': 'gemini-1.5-flash', 'temperature': 0.3}


In [76]:
df = pd.read_csv("data/transcripts.csv")

print("Data Loaded")
df.head()

Data Loaded


,call_id,agent_name,transcript,expected_call_type
0,1,Alice,Customer: I was charged twice for my premium t...,billing
1,2,Bob,Customer: My claim has been pending for 2 week...,claims
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query
4,5,Ethan,Customer: I noticed two charges for my monthly...,billing


In [77]:
df.tail(2)

,call_id,agent_name,transcript,expected_call_type
8,9,Isaac,Customer: I need a copy of the receipt for my ...,billing
9,10,Jasmine,Customer: My reimbursement claim has been wait...,claims


In [78]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def load_llm(config):
  provider = os.getenv("LLM_PROVIDER")
  if provider == "openai":
    llm = ChatOpenAI(
        model=config["llm"]["openai_model"],
        temperature=config["llm"]["temperature"]
    )
  elif provider == "google":
    llm = ChatGoogleGenerativeAI(
        model=config["llm"]["gemini_model"],
        temperature=config["llm"]["temperature"]
    )
  else:
    raise ValueError("Invalid provider")
  return llm

llm = load_llm(config)
print(f"LLM Loaded for provider: {os.getenv("LLM_PROVIDER")}")


LLM Loaded for provider: openai


In [79]:
response = llm.invoke("Say 'Setup Successful' in one short senence")
print(f"Printing LLM response: {response.content}")

Printing LLM response: Setup successful!


In [80]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define the output schema
class ClassificationOutput(BaseModel):
  call_type: str = Field(description = "Type of the call, e.g. 'Sales', 'Support', 'Complaint', etc.")
  confidence: float = Field(description = "Confidence score of the classification, between 0 and 1.")

parser = PydanticOutputParser(pydantic_object=ClassificationOutput)

#Prompt template for call type classification
prompt = PromptTemplate(
    template =
    """"
    You are a call classification helper. Classify the following call transcript into one of the following categories:
    {labels}

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
      "format_instructions": parser.get_format_instructions(),
      "labels": config["classification"]["labels"]
    }
)


#Create chain for call type classification
classification_chain = prompt | llm | parser

sample_text = df.iloc[0]["transcript"]
result = classification_chain.invoke({
  "transcript": sample_text
})


print(f"Classification Result: \n{result}")



Classification Result: 
call_type='billing' confidence=0.95


In [81]:
from tqdm import tqdm

result = []
print(df.__len__())

for i, row in tqdm(df.iterrows(), total=len(df), desc = "Classifying calls"):
  try:
    output = classification_chain.invoke({
      "transcript": row["transcript"]
    })

    result.append({
      "call_id": row["call_id"],
      "prediction": output.call_type,
      "confidence": output.confidence
    })
  except Exception as e:
    print(f"Error processing the row {i}: {e}")
    result.append({
      "call_id": row["call_id"],
      "prediction": None,
      "confidence": None
    })

results_df = pd.DataFrame(result)
df = df.merge(results_df, on="call_id")
df.head(100)

10


Classifying calls: 100%|██████████| 10/10 [00:09<00:00,  1.02it/s]


,call_id,agent_name,transcript,expected_call_type,prediction,confidence
0,1,Alice,Customer: I was charged twice for my premium t...,billing,billing,0.95
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,claims,0.95
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,complaint,0.85
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,general_query,0.85
4,5,Ethan,Customer: I noticed two charges for my monthly...,billing,billing,0.90
5,6,Fiona,Customer: I paid my bill last week but it stil...,billing,billing,0.90
6,7,George,Customer: Could you tell me when my next premi...,billing,billing,0.95
7,8,Hannah,Customer: I was charged the old rate after my ...,billing,billing,0.90
8,9,Isaac,Customer: I need a copy of the receipt for my ...,billing,billing,0.90
9,10,Jasmine,Customer: My reimbursement claim has been wait...,claims,claims,0.95


In [82]:
# Define routing logic based on call type

def route_call(call_type):
  if call_type == "billing":
    return ["knowledge_accuracy", "resolution_quality"]
  elif call_type == "claims":
    return ["resolution_quality", "knowledge_accuracy"]
  elif call_type == "complaint":
    return ["tone_empathy", "resolution_quality"]
  elif call_type == "general_query":
    return ["knowledge_accuracy"]
  else:
    return ["knowledge_accuracy"]

df["evolution_plan"] = df["prediction"].apply(route_call)

df.head(100)

,call_id,agent_name,transcript,expected_call_type,prediction,confidence,evolution_plan
0,1,Alice,Customer: I was charged twice for my premium t...,billing,billing,0.95,"[knowledge_accuracy, resolution_quality]"
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,claims,0.95,"[resolution_quality, knowledge_accuracy]"
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,complaint,0.85,"[tone_empathy, resolution_quality]"
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,general_query,0.85,[knowledge_accuracy]
4,5,Ethan,Customer: I noticed two charges for my monthly...,billing,billing,0.90,"[knowledge_accuracy, resolution_quality]"
5,6,Fiona,Customer: I paid my bill last week but it stil...,billing,billing,0.90,"[knowledge_accuracy, resolution_quality]"
6,7,George,Customer: Could you tell me when my next premi...,billing,billing,0.95,"[knowledge_accuracy, resolution_quality]"
7,8,Hannah,Customer: I was charged the old rate after my ...,billing,billing,0.90,"[knowledge_accuracy, resolution_quality]"
8,9,Isaac,Customer: I need a copy of the receipt for my ...,billing,billing,0.90,"[knowledge_accuracy, resolution_quality]"
9,10,Jasmine,Customer: My reimbursement claim has been wait...,claims,claims,0.95,"[resolution_quality, knowledge_accuracy]"


In [83]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# define an output schema for tone evaluation
class ToneEvaluation(BaseModel):
    score: int = Field(description = "Score between 1 and 5")
    reasoning: str = Field(description = "Explanation of the score")

tone_parser = PydanticOutputParser(pydantic_object = ToneEvaluation)

tone_prompt = PromptTemplate(
    template = """
    You are a QA evaluator for customer support calls.

    Evaluate the agent's tone and empathy in the following transcript.

    Consider:
    - Did the agent acknowledge the customer's issue?
    - Was the tone polite and professional enough?
    - Did the agent show empathy?

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": tone_parser.get_format_instructions()
    }

)

tone_chain = tone_prompt | llm | tone_parser

sample_text = df[df["prediction"] == "complaint"].iloc[0]["transcript"]

result = tone_chain.invoke({
    "transcript": sample_text
})

print(f"Tone evaluation result: \n{result}")

Tone evaluation result: 
score=3 reasoning="The agent acknowledged the customer's issue by apologizing for the inconvenience, which indicates some level of recognition of the customer's frustration. However, the response was quite brief and could have included more empathetic language to show understanding of the customer's repeated attempts to resolve the issue. The tone was polite, but it lacked warmth and a deeper connection to the customer's feelings."


In [84]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


# --- Step 1: Schema ---
class ResolutionEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionEvaluation)

# --- Step 2: Prompt ---
resolution_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the resolution quality of the agent.

Consider:
- Did the agent fully resolve the customer's issue?
- Were next steps clearly communicated?
- Did the agent confirm resolution before ending?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": resolution_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
resolution_chain = resolution_prompt | llm | resolution_parser

# --- Step 4: Test
sample_text = df.iloc[0]["transcript"]

result = resolution_chain.invoke({
    "transcript": sample_text
})

print("✅ Resolution Evaluation Result:")
print(result)

✅ Resolution Evaluation Result:
score=2 reasoning="The agent did not fully resolve the customer's issue as the transcript ends abruptly without confirming the resolution or providing next steps. There is no indication that the agent addressed the double charge issue."


In [85]:
# --- Step 1: Imports (safe to repeat) ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# --- Step 2: Schema ---
class KnowledgeEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

knowledge_parser = PydanticOutputParser(pydantic_object=KnowledgeEvaluation)

# --- Step 3: Prompt ---
knowledge_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's knowledge accuracy and clarity.

Consider:
- Did the agent provide correct and relevant information?
- Was the explanation clear and easy to understand?
- Did the agent avoid vague or misleading statements?

IMPORTANT:
- If the transcript does not contain enough information, give a moderate score (2 or 3) and explain why.

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": knowledge_parser.get_format_instructions()
    }
)

# --- Step 4: Chain ---
knowledge_chain = knowledge_prompt | llm | knowledge_parser

# --- Step 4: Test
sample_text = df.iloc[0]["transcript"]

result = knowledge_chain.invoke({
    "transcript": sample_text
})

print("✅ Knowledge Chain Ready")
print(f"Result from knowledge chain::: {result}")

✅ Knowledge Chain Ready
Result from knowledge chain::: score=3 reasoning="The agent's response indicates they are checking the issue, but there is no information provided about the agent's knowledge or the resolution of the customer's concern. The lack of a complete response makes it difficult to assess the accuracy and clarity of the information given."


In [86]:
from tqdm import tqdm

def run_evaluations(transcript, eval_plan):
  results = {}

  if "tone_empathy" in eval_plan:
    try:
      tone_result = tone_chain.invoke({"transcript": transcript})
      results["tone"] = tone_result.model_dump()
    except Exception as e:
      results["tone"] = {"error": str(e)}

  if "knowledge_accuracy" in eval_plan:
    try:
      knowledge_result = knowledge_chain.invoke({"transcript": transcript})
      results["knowledge"] = knowledge_result.model_dump()
    except Exception as e:
      results["knowledge"] = {"error": str(e)}

  if "resolution_empathy" in eval_plan:
    try:
      resolution_result = resolution_chain.invoke({"transcript": transcript})
      results["resolution"] = resolution_result.model_dump()
    except Exception as e:
      results["resolution"] = {"error": str(e)}
  return results


evaluation_outputs = []
for i, row in tqdm(df.iterrows(), total=len(df), desc="Running Evaluations"):
  output = run_evaluations(row["transcript"], row["evolution_plan"])

  evaluation_outputs.append(
      {
        "call_id": row["call_id"],
        "evaluation_output": output
      }
  )
# Convert to Dataframe
eval_df = pd.DataFrame(evaluation_outputs)

# Merge
df = df.merge(eval_df, on="call_id")

print("\n ✅ Evaluation completed")

df.head(100)


Running Evaluations: 100%|██████████| 10/10 [00:16<00:00,  1.66s/it]


 ✅ Evaluation completed


,call_id,agent_name,transcript,expected_call_type,prediction,confidence,evolution_plan,evaluation_output
0,1,Alice,Customer: I was charged twice for my premium t...,billing,billing,0.95,"[knowledge_accuracy, resolution_quality]","{'knowledge': {'score': 2, 'reasoning': 'The t..."
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,claims,0.95,"[resolution_quality, knowledge_accuracy]","{'knowledge': {'score': 2, 'reasoning': 'The t..."
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,complaint,0.85,"[tone_empathy, resolution_quality]","{'tone': {'score': 4, 'reasoning': 'The agent ..."
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,general_query,0.85,[knowledge_accuracy],"{'knowledge': {'score': 3, 'reasoning': 'The t..."
4,5,Ethan,Customer: I noticed two charges for my monthly...,billing,billing,0.90,"[knowledge_accuracy, resolution_quality]","{'knowledge': {'score': 3, 'reasoning': 'The a..."
5,6,Fiona,Customer: I paid my bill last week but it stil...,billing,billing,0.90,"[knowledge_accuracy, resolution_quality]","{'knowledge': {'score': 3, 'reasoning': 'The a..."
6,7,George,Customer: Could you tell me when my next premi...,billing,billing,0.95,"[knowledge_accuracy, resolution_quality]","{'knowledge': {'score': 3, 'reasoning': 'The a..."
7,8,Hannah,Customer: I was charged the old rate after my ...,billing,billing,0.90,"[knowledge_accuracy, resolution_quality]","{'knowledge': {'score': 3, 'reasoning': 'The a..."
8,9,Isaac,Customer: I need a copy of the receipt for my ...,billing,billing,0.90,"[knowledge_accuracy, resolution_quality]","{'knowledge': {'score': 3, 'reasoning': 'The a..."
9,10,Jasmine,Customer: My reimbursement claim has been wait...,claims,claims,0.95,"[resolution_quality, knowledge_accuracy]","{'knowledge': {'score': 3, 'reasoning': 'The a..."


In [87]:
# Show one example clearly
import pprint
pprint.pprint(df.iloc[0]["evaluation_output"])


{'knowledge': {'reasoning': 'The transcript does not provide enough '
                            "information about the agent's response to the "
                            "customer's issue regarding being charged twice. "
                            'While the agent indicates they will check the '
                            'issue, there is no further detail or resolution '
                            'provided. This lack of information leads to a '
                            'moderate score.',
               'score': 2}}


In [88]:
# --- Step 1: Schema ---
class FinalReport(BaseModel):
    summary: str = Field(description="Overall evaluation summary")
    recommendations: list[str] = Field(description="List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object=FinalReport)

# --- Step 2: Prompt ---
final_prompt = PromptTemplate(
    template="""
You are a QA manager reviewing customer support calls.

Based on the evaluation results below, generate:

1. A concise summary of the agent's performance
2. A list of actionable recommendations for improvement

Evaluation Data:
{evaluation_output}

IMPORTANT:
- Be specific and practical
- Do not repeat scores
- Focus on improvement

{format_instructions}
""",
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
final_chain = final_prompt | llm | final_parser

# --- Step 4: Test on one row
sample_eval = df.iloc[0]["evaluation_output"]

result = final_chain.invoke({
    "evaluation_output": sample_eval
})

print("✅ Final QA Report:")
print(result)

✅ Final QA Report:
summary="The agent demonstrated a basic understanding of the customer's issue regarding double charging but failed to provide a clear resolution or sufficient detail in their response. This resulted in a moderate performance evaluation due to the lack of follow-up information." recommendations=['Enhance knowledge of common billing issues to provide more detailed responses.', 'Encourage agents to offer specific next steps or timelines when addressing customer concerns.', 'Implement training sessions focused on effective communication and resolution strategies for billing inquiries.']


In [89]:
from tqdm import tqdm

final_outputs = []
for i, row in tqdm(df.iterrows(), total=len(df), desc="Generating Final Reports"):
  try:
    result = final_chain.invoke(
        {
          "evaluation_output": row["evaluation_output"]

        }
    )

    final_outputs.append({
      "call_id": row["call_id"],
      "summary": result.summary,
      "recommendations": result.recommendations
    })
  except Exception as e:
    print(f"👎🏻Error occuared at {i} --Error Details {e}")

# Created the dataframe with the end result
final_df = pd.DataFrame(final_outputs)
df = df.merge(final_df, on="call_id")
df[[
  "call_id", "prediction", "evaluation_output", "summary", "recommendations"
]]





Generating Final Reports: 100%|██████████| 10/10 [00:20<00:00,  2.07s/it]


,call_id,prediction,evaluation_output,summary,recommendations
0,1,billing,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent demonstrated a basic understanding o...,[Enhance knowledge of common billing issues to...
1,2,claims,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent demonstrated a lack of clarity and r...,[Enhance product knowledge by reviewing common...
2,3,complaint,"{'tone': {'score': 4, 'reasoning': 'The agent ...",The agent demonstrated a polite and profession...,[Enhance empathetic communication by acknowled...
3,4,general_query,"{'knowledge': {'score': 3, 'reasoning': 'The t...",The agent demonstrated a willingness to explai...,[Provide specific examples and details when ex...
4,5,billing,"{'knowledge': {'score': 3, 'reasoning': 'The a...",The agent demonstrated a willingness to assist...,[Enhance product knowledge training to ensure ...
5,6,billing,"{'knowledge': {'score': 3, 'reasoning': 'The a...",The agent demonstrated a basic understanding o...,[Enhance knowledge of the payment process to p...
6,7,billing,"{'knowledge': {'score': 3, 'reasoning': 'The a...",The agent demonstrated a moderate understandin...,[Familiarize with the specific details of the ...
7,8,billing,"{'knowledge': {'score': 3, 'reasoning': 'The a...",The agent demonstrated an attempt to engage wi...,[Enhance knowledge of all current plans and th...
8,9,billing,"{'knowledge': {'score': 3, 'reasoning': 'The a...",The agent demonstrated a willingness to assist...,[Provide step-by-step instructions when assist...
9,10,claims,"{'knowledge': {'score': 3, 'reasoning': 'The a...",The agent demonstrated an understanding of the...,[Provide detailed information about the reimbu...


In [90]:
accuracy = (df["expected_call_type"] == df["prediction"]).mean()
print(f"Classification Accuracy: {accuracy}")

Classification Accuracy: 1.0


In [91]:
df.to_excel("data/output.xlsx", index = False)